# Module 2: Directed Acyclic Graphs and the Backdoor Criterion

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

A DAG is a picture of what causes what. It is not a statistical model and it
is not estimated from the data: it is drawn from what is known about how the
world produced these records.

Its value is that once it is drawn, **the set of variables to condition on can
be read off it mechanically**, before any model is fitted, and so can the set
that must be left alone.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

KEEP = [a for a in TRAINED if a != "A007"]     # the pre trend violator, Intermediate 8
BASELINE = profile.set_index("agency_id")["pre_program_uof_per_100_arrests"]

d = f[f["agency_id"] != "A007"].copy()
d["lo"] = np.log(d["n_arrests"])
pi = pd.PeriodIndex(d["year_month"], freq="M")
d["yr"] = pi.year.values + (pi.month.values - 1) / 12.0
d["base"] = d["agency_id"].map(BASELINE)

pct = lambda b: 100 * (np.exp(b) - 1)


def fit(data, treated, form=None, outcome="n_uof", offset=None):
    s = data.copy()
    s["settled"] = ((s["agency_id"].isin(treated))
                    & (s["period"] == "after")).astype(float)
    s["phase"] = ((s["agency_id"].isin(treated))
                  & (s["period"] == "phase")).astype(float)
    fo = form or f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase"
    z = smf.glm(fo, s, family=sm.families.Poisson(),
                offset=s["lo"] if offset is None else offset).fit()
    lo, hi = z.conf_int().loc["settled"]
    return pct(z.params["settled"]), pct(lo), pct(hi), z

## 2. The graph for this study

Four nodes are enough.

| Node | Meaning |
|---|---|
| **D** | the agency took the de escalation training |
| **Y** | use of force per arrest |
| **B** | the agency's baseline use of force rate |
| **T** | the statewide year, standing for everything moving over time |

And five edges, each of which comes from documented knowledge rather than
from the data:

> **D → Y** the effect to be identified
>
> **B → D** the selection rule used the baseline rate
>
> **B → Y** an agency with a high baseline has a high rate later
>
> **T → Y** everything was falling statewide
>
> **T → D** the program was adopted at a particular time, when rates were already lower

## 3. Reading the backdoor paths

A **backdoor path** is any path from D to Y that starts with an arrow **into**
D. It carries association that is not the effect.

There are two:

> D ← B → Y
>
> D ← T → Y

The **backdoor criterion** says: a set of variables identifies the effect if
it blocks every backdoor path and contains no descendant of D.

Both paths here are simple chains through a common cause, so conditioning on
B and on T blocks them.

## 4. Checking that by fitting, which is the wrong order but instructive

The point of the graph is that this table could have been written **before**
any of these models were run.

In [ ]:
specs = [
    ("nothing", "n_uof ~ settled + phase", "both paths open"),
    ("B only", "n_uof ~ base + settled + phase", "T still open"),
    ("T only", "n_uof ~ C(year_month) + settled + phase", "B still open"),
    ("B and T", "n_uof ~ base + C(year_month) + settled + phase", "both closed"),
    ("agency and month effects",
     "n_uof ~ C(agency_id) + C(year_month) + settled + phase", "both closed"),
]
rows = []
for lab, form, note in specs:
    e, lo, hi, _ = fit(d, KEEP, form=form)
    rows.append({"conditioned on": lab, "estimate": f"{e:+.1f}%",
                 "95 percent interval": f"[{lo:+.1f}, {hi:+.1f}]",
                 "what the graph says is left open": note})
rows.append({"conditioned on": "THE TRUTH", "estimate": f"{TRUTH:+.1f}%",
             "95 percent interval": "", "what the graph says is left open": ""})
pd.DataFrame(rows).set_index("conditioned on")

The two sets the graph says are sufficient land on the truth. The three it
says are insufficient do not, and **the one conditioning only on time gets the
sign wrong**: with the baseline path open, the treated agencies' higher levels
show up as a positive coefficient.

Agency fixed effects work because agency identity determines the baseline
rate, so conditioning on the agency conditions on B and on anything else fixed
about the agency. That is strictly more than the graph requires, and it is
cheap.

## 5. What the graph forbids

The criterion has a second half that is easier to violate: **no descendant of
D.** Anything caused by the training is off limits.

In [ ]:
s = d.copy()
s["settled"] = ((s["agency_id"].isin(KEEP)) & (s["period"] == "after")).astype(float)
s["phase"] = ((s["agency_id"].isin(KEEP)) & (s["period"] == "phase")).astype(float)
for label, outcome in [("arrests", "n_arrests"), ("calls for service", "total_cfs")]:
    z = smf.glm(f"{outcome} ~ C(agency_id)+C(year_month)+settled+phase", s,
                family=sm.families.Poisson()).fit()
    lo, hi = z.conf_int().loc["settled"]
    verdict = "a descendant of D" if not (lo < 0 < hi) else "not a descendant"
    print(f"  did the program move {label:18s} "
          f"{pct(z.params['settled']):+6.2f}%  [{pct(lo):+.2f}, {pct(hi):+.2f}]"
          f"   {verdict}")

Arrests are untouched, so using them as the denominator does not condition on
a descendant.

Calls for service move by 0.42 percent with an interval excluding zero. By the
strict reading that makes them a descendant and therefore forbidden as a
control. By any substantive reading a 0.42 percent movement is not a causal
pathway. **The graph is a tool for thinking, not an oracle**, and a 0.42
percent edge is a reason to note the ambiguity rather than to restructure the
analysis.

## 6. What a DAG cannot do

| It can | It cannot |
|---|---|
| tell you what to condition on, given the structure | tell you the structure |
| show that a design is unidentified | show that it is identified in reality |
| make an assumption visible and arguable | make it true |
| rule a control variable out | rule one in without the rest of the graph |

**The graph is an argument, and it is falsifiable only where it implies
testable restrictions.** Drawing one does not add information. It makes
whatever you already believed explicit enough to disagree with, which is most
of its value.

## Exercise

Add a node. Suppose agencies with reform minded leadership were both more
likely to adopt the program and independently improving. Where does it go, and
what does it do?

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    print("""
    The new node L sits exactly where B does:

        L -> D     reform minded leaders adopt the program
        L -> Y     they change other things that lower use of force

    That is a third backdoor path, D <- L -> Y, and the criterion says to
    condition on L.

    The difficulty is that L is not in any file. Agency fixed effects absorb
    it only if leadership is fixed over the study period, which for a seven
    year window is exactly the assumption in doubt: a chief who arrived in
    2022 and then adopted the program is a within agency change.
    """)
    pre = d[d["period"] == "before"].copy()
    pre["tr"] = pre["agency_id"].isin(KEEP).astype(float)
    z = smf.glm("n_uof ~ C(agency_id) + yr + tr:yr", pre,
                family=sm.families.Poisson(), offset=pre["lo"]).fit()
    k = [x for x in z.params.index if "yr" in x and "tr" in x][0]
    lo, hi = z.conf_int().loc[k]
    print(f"    the testable implication: if L were already acting before the")
    print(f"    program, the treated agencies would already be improving faster.")
    print(f"      difference in pre trends {pct(z.params[k]):+.2f}% a year  "
          f"[{pct(lo):+.2f}, {pct(hi):+.2f}]")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

L is an unmeasured confounder in exactly the position of B, and no adjustment
set built from the recorded variables blocks it.

**What the graph buys is the testable implication.** If reform minded
leadership was already acting before the program, the treated agencies should
already have been improving faster, and the pre trend comparison is the test.
It comes back at 0.70 percent a year with an interval reaching 3.31, which is
the same answer as Intermediate Module 15 and carries the same caveat: the
test does not rule the confounder out, it bounds how much of it the data can
see.

This is the honest use of a DAG on observational data. It does not deliver
identification. It tells you which assumption you are relying on, points at
the check that bears on it, and makes the residual risk a stated quantity
rather than an unstated one.

</details>

---

**Next:** [Module 3: Identification Before Estimation](Module_03_Identification_Before_Estimation.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*